# 06. Transcripción automática segmentada + subida a GCS

Este notebook transcribe los segmentos diarizados con `faster-whisper`. Guarda outputs locales por audio y, además, sube checkpoints y resultados oficiales al bucket para poder reanudar si la VM se cierra.

In [1]:
%pip install -q faster-whisper

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from IPython.display import clear_output, display
import os
import gc
import time
import re
import shutil

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

from faster_whisper import WhisperModel

print("Imports cargados correctamente.")

try:
    from google.cloud import storage
    HAS_GCP = True
except Exception:
    HAS_GCP = False
    storage = None

print("GCP disponible:", HAS_GCP)


Imports cargados correctamente.
GCP disponible: True


**Rutas principales:** aquí conectamos este notebook con lo que creamos en el notebook anterior

In [3]:
from pathlib import Path

PROJECT_DIR = Path("/home/jupyter/TFM_ProcesadoDeAudios")

# IMPORTANTE:
# Este CSV debe ser el consolidado corregido del notebook 05.
# Esperado tras corrección: 1.191 audios con segmentos finales válidos.
SEGMENTS_CSV = PROJECT_DIR / "data" / "diarization_outputs" / "consolidated" / "all_final_merged_segments.csv"

# Carpetas donde pueden estar los audios limpios
AUDIO_DIRS = [
    PROJECT_DIR / "data" / "clean_audios",
    PROJECT_DIR / "data" / "diarization_input_clean_audios",
]

TRANSCRIPTION_DIR = PROJECT_DIR / "data" / "transcription_outputs"
PER_AUDIO_DIR = TRANSCRIPTION_DIR / "per_audio"

TRANSCRIPTION_DIR.mkdir(parents=True, exist_ok=True)
PER_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

TRANSCRIBED_SEGMENTS_CSV = TRANSCRIPTION_DIR / "all_segments_transcribed.csv"
TRANSCRIPTION_SUMMARY_CSV = TRANSCRIPTION_DIR / "transcription_summary.csv"

SAVE_OUTPUTS_TO_GCS = True
UPLOAD_PER_AUDIO_TO_GCS = True
UPLOAD_FINALS_TO_GCS = True

# Limpieza de duplicados/stale outputs.
# Esto borra SOLO outputs de transcripción que no estén en el universo corregido de audio_id_base.
DELETE_STALE_LOCAL_TRANSCRIPTIONS = True
DELETE_STALE_GCS_TRANSCRIPTIONS = True
CANONICALIZE_LOCAL_TRANSCRIPTIONS = True

EXPECTED_FINAL_AUDIO_IDS = 1191

GCS_OUTPUT_PREFIX = "gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/"
GCS_PER_AUDIO_PREFIX = GCS_OUTPUT_PREFIX.rstrip("/") + "/per_audio/"

print("SEGMENTS_CSV:", SEGMENTS_CSV)
print("AUDIO_DIRS:")
for d in AUDIO_DIRS:
    print("-", d, "| existe:", d.exists())
print("TRANSCRIPTION_DIR:", TRANSCRIPTION_DIR)
print("PER_AUDIO_DIR:", PER_AUDIO_DIR)
print("GCS_OUTPUT_PREFIX:", GCS_OUTPUT_PREFIX)
print("GCS_PER_AUDIO_PREFIX:", GCS_PER_AUDIO_PREFIX)


SEGMENTS_CSV: /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_outputs/consolidated/all_final_merged_segments.csv
AUDIO_DIRS:
- /home/jupyter/TFM_ProcesadoDeAudios/data/clean_audios | existe: True
- /home/jupyter/TFM_ProcesadoDeAudios/data/diarization_input_clean_audios | existe: True
TRANSCRIPTION_DIR: /home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs
PER_AUDIO_DIR: /home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/per_audio
GCS_OUTPUT_PREFIX: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/
GCS_PER_AUDIO_PREFIX: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/per_audio/


In [4]:
# =========================
# HELPERS GCS
# =========================

def split_gcs_uri(uri: str):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"URI GCS inválida: {uri}")
    path = str(uri)[5:]
    bucket_name, _, prefix = path.partition("/")
    return bucket_name, prefix


def upload_file_to_gcs(local_path, gcs_prefix=GCS_OUTPUT_PREFIX):
    """Subida segura. Si falla GCS, avisa y el notebook sigue."""
    if not SAVE_OUTPUTS_TO_GCS:
        return None
    if not HAS_GCP:
        print("GCP no disponible; no se sube:", local_path)
        return None
    local_path = Path(local_path)
    if not local_path.exists():
        print("No existe:", local_path)
        return None
    try:
        bucket_name, prefix = split_gcs_uri(gcs_prefix)
        blob_path = f"{prefix.rstrip('/')}/{local_path.name}"
        client = storage.Client()
        client.bucket(bucket_name).blob(blob_path).upload_from_filename(str(local_path))
        return f"gs://{bucket_name}/{blob_path}"
    except Exception as e:
        print(f"Error subiendo {local_path.name} a GCS: {e}")
        return None

In [5]:
# =========================
# NORMALIZACIÓN DE IDs Y LIMPIEZA SEGURA
# =========================

def normalize_audio_id(x):
    """
    Normaliza nombres de audio/CSV para evitar contar como distintos:
    raw_915..._clean.wav
    915..._clean.wav
    raw_bajas_915..._final_merged.csv
    915..._transcribed_segments.csv

    Devuelve el audio_id_base.
    """
    x = Path(str(x)).name

    # quitar extensiones
    x = re.sub(r"\.csv$", "", x)
    x = re.sub(r"\.wav$", "", x)

    # quitar sufijos de outputs
    x = re.sub(r"_final_merged$", "", x)
    x = re.sub(r"_transcribed_segments$", "", x)

    # quitar sufijo de audio limpio
    x = re.sub(r"_clean$", "", x)

    # quitar prefijos de ejecuciones/corpus
    prefixes = ["raw_bajas_", "raw_comercial_", "raw_", "bajas_", "comercial_"]
    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if x.startswith(prefix):
                x = x[len(prefix):]
                changed = True

    return x


def canonical_transcription_path(audio_id_base: str):
    return PER_AUDIO_DIR / f"{audio_id_base}_transcribed_segments.csv"


def get_transcription_csv_audio_id(file_path):
    return normalize_audio_id(Path(file_path).name)


def delete_gcs_blob_uri(gcs_uri):
    """Borra un objeto GCS exacto. Solo se usa dentro de prefijos controlados."""
    if not HAS_GCP:
        print("GCP no disponible; no se puede borrar:", gcs_uri)
        return False
    bucket_name, blob_path = split_gcs_uri(gcs_uri)
    client = storage.Client()
    blob = client.bucket(bucket_name).blob(blob_path)
    if blob.exists():
        blob.delete()
        return True
    return False


def list_gcs_blobs(gcs_prefix):
    """Lista blobs bajo un prefijo GCS."""
    if not HAS_GCP:
        print("GCP no disponible; no se puede listar:", gcs_prefix)
        return []
    bucket_name, prefix = split_gcs_uri(gcs_prefix)
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    return list(bucket.list_blobs(prefix=prefix))


**Cargar CSV de segmentos**

In [6]:
# =========================
# CARGA DE SEGMENTOS FINALES CORREGIDOS
# =========================

df_segments = pd.read_csv(SEGMENTS_CSV)

# Normalizamos el universo de audios para no volver a contar raw_/sin raw_ como distintos
df_segments["audio_id_base"] = df_segments["audio_file"].apply(normalize_audio_id)

print("Dimensiones:", df_segments.shape)
print("Audios únicos por audio_file:", df_segments["audio_file"].nunique())
print("Audios únicos por audio_id_base:", df_segments["audio_id_base"].nunique())
print("Segmentos totales:", len(df_segments))
print("Columnas:")
print(df_segments.columns.tolist())

if df_segments["audio_id_base"].nunique() != EXPECTED_FINAL_AUDIO_IDS:
    print(f"⚠️ OJO: se esperaban {EXPECTED_FINAL_AUDIO_IDS} audios finales válidos, pero hay {df_segments['audio_id_base'].nunique()}.")

display(df_segments.head())


Dimensiones: (40352, 26)
Audios únicos por audio_file: 1181
Audios únicos por audio_id_base: 1181
Segmentos totales: 40352
Columnas:
['segment_id_raw', 'audio_file', 'audio_stem', 'start', 'end', 'duration', 'speaker', 'rms_dbfs', 'overlap_duration_sec', 'overlap_ratio', 'valid_export', 'valid_anchor', 'drop_reasons', 'anchor_reasons', 'speaker_original', 'speaker_final', 'relabel_source', 'best_distance', 'second_distance', 'distance_margin', 'dist_SPEAKER_00', 'dist_SPEAKER_01', 'merged_n_segments', 'segment_ids_raw', 'source_csv', 'audio_id_base']
⚠️ OJO: se esperaban 1191 audios finales válidos, pero hay 1181.


,segment_id_raw,audio_file,audio_stem,start,end,duration,speaker,rms_dbfs,overlap_duration_sec,overlap_ratio,...,relabel_source,best_distance,second_distance,distance_margin,dist_SPEAKER_00,dist_SPEAKER_01,merged_n_segments,segment_ids_raw,source_csv,audio_id_base
0,1,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,0.030969,4.452219,4.421250,SPEAKER_01,-28.919975,0.000000,0.000000,...,centroid,0.145629,0.747864,0.602235,0.747864,0.145629,1,1,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
1,2,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,5.211594,6.342219,1.130625,SPEAKER_00,-33.883114,0.000000,0.000000,...,centroid,0.607482,0.814511,0.207029,0.607482,0.814511,1,2,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
2,3,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,6.426594,13.800969,7.374375,SPEAKER_01,-29.186810,0.000000,0.000000,...,centroid,0.071406,0.732181,0.660775,0.732181,0.071406,1,3,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
3,4,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,13.800969,14.965344,1.164375,SPEAKER_00,-36.389011,0.118125,0.101449,...,centroid,0.689627,0.975324,0.285697,0.689627,0.975324,1,4,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851
4,5,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,15.285969,19.994094,4.708125,SPEAKER_01,-27.753717,0.000000,0.000000,...,centroid,0.203869,0.709531,0.505662,0.709531,0.203869,1,5,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851


In [7]:
required_cols = ["audio_file", "audio_id_base", "start", "end", "duration", "speaker_final"]

missing_cols = [col for col in required_cols if col not in df_segments.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas requeridas: {missing_cols}")

print("Columnas requeridas OK.")


Columnas requeridas OK.


**Comprobamos que encuentra los audios**

In [8]:
# =========================
# COMPROBACIÓN ROBUSTA DE AUDIOS LOCALES
# =========================

def get_audio_name_candidates(audio_file: str, audio_id_base: str = None):
    """
    Genera posibles nombres del audio limpio, porque algunos outputs aparecen
    con prefijo raw_ y otros sin prefijo.
    """
    audio_file = str(audio_file)
    audio_id_base = normalize_audio_id(audio_file) if audio_id_base is None else str(audio_id_base)

    candidates = [
        audio_file,
        f"{audio_id_base}_clean.wav",
        f"raw_{audio_id_base}_clean.wav",
        f"raw_bajas_{audio_id_base}_clean.wav",
        f"raw_comercial_{audio_id_base}_clean.wav",
        f"{audio_id_base}.wav",
        f"raw_{audio_id_base}.wav",
        f"raw_bajas_{audio_id_base}.wav",
        f"raw_comercial_{audio_id_base}.wav",
    ]

    return list(dict.fromkeys(candidates))


def resolve_audio_path(audio_file: str, audio_id_base: str = None):
    """
    Busca el audio en todas las carpetas posibles y con variantes de nombre.
    """
    audio_id_base = normalize_audio_id(audio_file) if audio_id_base is None else str(audio_id_base)

    for audio_dir in AUDIO_DIRS:
        for candidate_name in get_audio_name_candidates(audio_file, audio_id_base):
            candidate_path = audio_dir / candidate_name
            if candidate_path.exists():
                return candidate_path

    return None


df_segments["resolved_audio_path"] = df_segments.apply(
    lambda r: resolve_audio_path(r["audio_file"], r["audio_id_base"]),
    axis=1
)
df_segments["audio_exists_local"] = df_segments["resolved_audio_path"].notna()

print("Audios esperados por audio_id_base:", df_segments["audio_id_base"].nunique())
print("Audios encontrados localmente:", df_segments.loc[df_segments["audio_exists_local"], "audio_id_base"].nunique())
print("Segmentos con audio localizado:", df_segments["audio_exists_local"].sum())
print("Segmentos sin audio localizado:", (~df_segments["audio_exists_local"]).sum())

missing_audio = (
    df_segments.loc[~df_segments["audio_exists_local"], ["audio_file", "audio_id_base"]]
    .drop_duplicates()
)

if len(missing_audio) > 0:
    display(missing_audio.head(30))
else:
    print("Todos los audios han sido localizados correctamente.")


Audios esperados por audio_id_base: 1181
Audios encontrados localmente: 1181
Segmentos con audio localizado: 40352
Segmentos sin audio localizado: 0
Todos los audios han sido localizados correctamente.


In [9]:
# =========================
# LIMPIEZA Y CANONICALIZACIÓN DE OUTPUTS DE TRANSCRIPCIÓN
# =========================
# Objetivo:
# - eliminar outputs locales que no pertenezcan al universo corregido de 1.191 audios;
# - evitar que archivos antiguos raw_/duplicados vuelvan a subirse a GCS;
# - conservar, si existe, una transcripción válida por audio_id_base para no retranscribir innecesariamente.

target_audio_ids = set(df_segments["audio_id_base"].dropna().astype(str).unique())

print("Audios objetivo para transcripción:", len(target_audio_ids))

def audit_transcription_file(f):
    row = {
        "path": str(f),
        "file": f.name,
        "audio_id_base": get_transcription_csv_audio_id(f),
        "mtime": f.stat().st_mtime,
        "exists": f.exists(),
        "n_segments": 0,
        "n_ok": 0,
        "n_text": 0,
        "text_ratio": 0.0,
        "statuses": "",
        "is_bad": True,
    }
    try:
        df_tmp = pd.read_csv(f)
        row["n_segments"] = len(df_tmp)
        if len(df_tmp) > 0:
            if "audio_file" in df_tmp.columns:
                # Preferimos el audio_id_base real del contenido, si existe.
                row["audio_id_base"] = normalize_audio_id(df_tmp["audio_file"].iloc[0])
            if "transcription_status" in df_tmp.columns:
                row["n_ok"] = int((df_tmp["transcription_status"] == "ok").sum())
                row["statuses"] = ", ".join(sorted(df_tmp["transcription_status"].dropna().astype(str).unique()))
            if "text" in df_tmp.columns:
                row["n_text"] = int(df_tmp["text"].fillna("").astype(str).str.strip().ne("").sum())
            row["text_ratio"] = row["n_text"] / row["n_segments"] if row["n_segments"] > 0 else 0.0

        row["is_bad"] = (
            row["n_segments"] == 0
            or "audio_not_found" in row["statuses"]
            or row["n_ok"] == 0
            or row["text_ratio"] < 0.20
        )
    except Exception as e:
        row["statuses"] = f"read_error: {e}"
        row["is_bad"] = True
    return row


local_files = sorted(PER_AUDIO_DIR.glob("*_transcribed_segments.csv"))
df_local_audit = pd.DataFrame([audit_transcription_file(f) for f in local_files])

print("Outputs locales encontrados:", len(df_local_audit))

if len(df_local_audit) > 0:
    df_local_audit["in_target"] = df_local_audit["audio_id_base"].isin(target_audio_ids)
    display(
        df_local_audit
        .groupby(["in_target", "is_bad"])
        .size()
        .reset_index(name="n_files")
    )

    # 1) borrar stale o malos
    to_delete = df_local_audit[
        (~df_local_audit["in_target"]) | (df_local_audit["is_bad"])
    ].copy()

    if DELETE_STALE_LOCAL_TRANSCRIPTIONS:
        deleted = 0
        for p in to_delete["path"]:
            p = Path(p)
            if p.exists():
                p.unlink()
                deleted += 1
        print("Outputs locales eliminados por stale/malos:", deleted)
    else:
        print("DELETE_STALE_LOCAL_TRANSCRIPTIONS=False; no se borró nada local.")

    # 2) canonicalizar duplicados buenos dentro del universo objetivo
    df_good = df_local_audit[
        (df_local_audit["in_target"])
        & (~df_local_audit["is_bad"])
        & (~df_local_audit["path"].isin(to_delete["path"]))
    ].copy()

    if CANONICALIZE_LOCAL_TRANSCRIPTIONS and len(df_good) > 0:
        keep_rows = (
            df_good
            .sort_values(
                ["audio_id_base", "text_ratio", "n_ok", "n_segments", "mtime"],
                ascending=[True, False, False, False, False]
            )
            .drop_duplicates("audio_id_base", keep="first")
        )

        keep_paths = set(keep_rows["path"])
        copied = 0
        removed_dupes = 0

        for _, r in keep_rows.iterrows():
            src_path = Path(r["path"])
            dst_path = canonical_transcription_path(r["audio_id_base"])
            if src_path != dst_path:
                shutil.copy2(src_path, dst_path)
                copied += 1

        # borrar duplicados no elegidos y no canónicos
        for p in df_good.loc[~df_good["path"].isin(keep_paths), "path"]:
            p = Path(p)
            if p.exists():
                p.unlink()
                removed_dupes += 1

        # borrar archivos elegidos no canónicos después de copiar
        for _, r in keep_rows.iterrows():
            src_path = Path(r["path"])
            dst_path = canonical_transcription_path(r["audio_id_base"])
            if src_path != dst_path and src_path.exists():
                src_path.unlink()
                removed_dupes += 1

        print("Outputs copiados a nombre canónico:", copied)
        print("Duplicados/no canónicos eliminados:", removed_dupes)

# Auditoría final local
local_files_after = sorted(PER_AUDIO_DIR.glob("*_transcribed_segments.csv"))
df_local_after = pd.DataFrame([audit_transcription_file(f) for f in local_files_after])

if len(df_local_after) > 0:
    df_local_after["in_target"] = df_local_after["audio_id_base"].isin(target_audio_ids)
    print("Outputs locales finales:", len(df_local_after))
    print("Outputs locales finales en universo objetivo:", int(df_local_after["in_target"].sum()))
    print("Audio_id_base únicos locales finales:", df_local_after["audio_id_base"].nunique())
    display(df_local_after.sort_values(["in_target", "is_bad"], ascending=[True, False]).head(20))
else:
    print("No hay outputs locales de transcripción después de la limpieza.")


# =========================
# LIMPIEZA SEGURA DE GCS /per_audio/
# =========================
# Borra del bucket los CSV de transcripción por audio que no pertenezcan al universo corregido
# o que tengan nombre no canónico. El prefijo está limitado a GCS_PER_AUDIO_PREFIX.

if DELETE_STALE_GCS_TRANSCRIPTIONS and SAVE_OUTPUTS_TO_GCS and HAS_GCP:
    print("Listando outputs GCS en:", GCS_PER_AUDIO_PREFIX)
    blobs = list_gcs_blobs(GCS_PER_AUDIO_PREFIX)

    gcs_rows = []
    for blob in blobs:
        name = Path(blob.name).name
        if not name.endswith("_transcribed_segments.csv"):
            continue
        audio_id_base = normalize_audio_id(name)
        canonical_name = f"{audio_id_base}_transcribed_segments.csv"
        is_canonical = (name == canonical_name)
        in_target = audio_id_base in target_audio_ids

        gcs_rows.append({
            "blob_name": blob.name,
            "file": name,
            "audio_id_base": audio_id_base,
            "in_target": in_target,
            "is_canonical": is_canonical,
            "delete": (not in_target) or (not is_canonical)
        })

    df_gcs_audit = pd.DataFrame(gcs_rows)

    print("CSVs de transcripción encontrados en GCS:", len(df_gcs_audit))
    if len(df_gcs_audit) > 0:
        print("A borrar de GCS:", int(df_gcs_audit["delete"].sum()))
        display(df_gcs_audit.groupby(["in_target", "is_canonical", "delete"]).size().reset_index(name="n_files"))

        bucket_name, prefix = split_gcs_uri(GCS_PER_AUDIO_PREFIX)
        client = storage.Client()
        bucket = client.bucket(bucket_name)

        deleted = 0
        for blob_name in df_gcs_audit.loc[df_gcs_audit["delete"], "blob_name"]:
            bucket.blob(blob_name).delete()
            deleted += 1
        print("Objetos eliminados de GCS:", deleted)
else:
    print("No se limpió GCS. Revisa DELETE_STALE_GCS_TRANSCRIPTIONS, SAVE_OUTPUTS_TO_GCS y HAS_GCP.")


Audios objetivo para transcripción: 1181
Outputs locales encontrados: 1181


,in_target,is_bad,n_files
0,True,False,1181


Outputs locales eliminados por stale/malos: 0
Outputs copiados a nombre canónico: 0
Duplicados/no canónicos eliminados: 0
Outputs locales finales: 1181
Outputs locales finales en universo objetivo: 1181
Audio_id_base únicos locales finales: 1181


,path,file,audio_id_base,mtime,exists,n_segments,n_ok,n_text,text_ratio,statuses,is_bad,in_target
0,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154117451310006851_transcribed_segments.csv,9154117451310006851,1.783172e+09,True,37,36,36,0.972973,"empty_transcription, ok",False,True
1,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154117551220006851_transcribed_segments.csv,9154117551220006851,1.783172e+09,True,41,41,41,1.000000,ok,False,True
2,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154127337680006851_transcribed_segments.csv,9154127337680006851,1.783172e+09,True,27,27,27,1.000000,ok,False,True
3,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154142438160016851_transcribed_segments.csv,9154142438160016851,1.783172e+09,True,46,45,45,0.978261,"empty_transcription, ok",False,True
4,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154152155960016851_transcribed_segments.csv,9154152155960016851,1.783172e+09,True,38,37,37,0.973684,"empty_transcription, ok",False,True
5,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154188548830006851_transcribed_segments.csv,9154188548830006851,1.783172e+09,True,23,23,23,1.000000,ok,False,True
6,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154202560160006851_transcribed_segments.csv,9154202560160006851,1.783172e+09,True,19,19,19,1.000000,ok,False,True
7,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154202749900006851_transcribed_segments.csv,9154202749900006851,1.783172e+09,True,30,29,29,0.966667,"empty_transcription, ok",False,True
8,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154219744450006851_transcribed_segments.csv,9154219744450006851,1.783172e+09,True,43,42,42,0.976744,"empty_transcription, ok",False,True
9,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,9154222339820016851_transcribed_segments.csv,9154222339820016851,1.783172e+09,True,50,50,50,1.000000,ok,False,True


Listando outputs GCS en: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/per_audio/
CSVs de transcripción encontrados en GCS: 1181
A borrar de GCS: 0


,in_target,is_canonical,delete,n_files
0,True,True,False,1181


Objetos eliminados de GCS: 0


**Cargar modelo de transcripción**

In [10]:
# =========================
# CARGA DEL MODELO WHISPER
# =========================

MODEL_SIZE = "base"   # opciones: "base", "small", "medium", "large-v3"

DEVICE = "cuda" if os.system("nvidia-smi > /dev/null 2>&1") == 0 else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

print("Device:", DEVICE)
print("Compute type:", COMPUTE_TYPE)
print("Modelo:", MODEL_SIZE)

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)

print("Modelo cargado correctamente.")

Device: cpu
Compute type: int8
Modelo: base
Modelo cargado correctamente.


**Funciones auxiliares:** para cargar audios, cortar segmentos y transcribirlos

In [11]:
# =========================
# FUNCIONES AUXILIARES
# =========================

TARGET_SR = 16000
LANGUAGE = "es"  # si hay mucho catalán/inglés mezclado, se puede poner None

def load_audio_mono(audio_path: Path, target_sr: int = TARGET_SR):
    """
    Carga un audio en mono y lo remuestrea a 16 kHz si es necesario.
    """
    audio, sr = sf.read(audio_path, always_2d=False)

    if isinstance(audio, np.ndarray) and audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    audio = audio.astype(np.float32)

    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    return audio, sr


def extract_segment_audio(audio: np.ndarray, sr: int, start: float, end: float):
    """
    Extrae un fragmento de audio usando start/end en segundos.
    """
    start_sample = max(0, int(round(start * sr)))
    end_sample = min(len(audio), int(round(end * sr)))

    if end_sample <= start_sample:
        return np.array([], dtype=np.float32)

    return audio[start_sample:end_sample].astype(np.float32)


def transcribe_audio_array(segment_audio: np.ndarray):
    """
    Transcribe un fragmento de audio con faster-whisper.
    """
    if segment_audio.size == 0:
        return "", "empty_audio"

    try:
        segments, info = model.transcribe(
            segment_audio,
            language=LANGUAGE,
            beam_size=5,
            vad_filter=False
        )

        text_parts = []

        for seg in segments:
            if seg.text is not None:
                text_parts.append(seg.text.strip())

        text = " ".join(text_parts).strip()

        if text == "":
            return "", "empty_transcription"

        return text, "ok"

    except Exception as e:
        return "", f"error: {str(e)}"

Hacemos una **prueba antes del batch** completo

In [12]:
# =========================
# PRUEBA ROBUSTA CON UN AUDIO
# =========================

# Escoger automáticamente un audio que sí tenga audio localizado
df_available = df_segments[df_segments["audio_exists_local"] == True].copy()

if df_available.empty:
    raise FileNotFoundError(
        "No hay audios localizados. Revisa la celda de COMPROBACIÓN ROBUSTA DE AUDIOS LOCALES."
    )

# Si quieres probar uno específico, puedes ponerlo aquí.
# Si lo dejas en None, escoge el primero disponible.
MANUAL_AUDIO_ID_BASE = None
# MANUAL_AUDIO_ID_BASE = "9154117451310006851"

if MANUAL_AUDIO_ID_BASE is not None:
    df_available = df_available[df_available["audio_id_base"] == MANUAL_AUDIO_ID_BASE].copy()
    
    if df_available.empty:
        raise FileNotFoundError(
            f"No se encontró audio localizado para audio_id_base={MANUAL_AUDIO_ID_BASE}"
        )

test_audio_id_base = df_available["audio_id_base"].iloc[0]
test_audio_file = df_available["audio_file"].iloc[0]
audio_path = resolve_audio_path(test_audio_file, test_audio_id_base)

if audio_path is None:
    raise FileNotFoundError(
        f"No se pudo resolver la ruta del audio para {test_audio_file} / {test_audio_id_base}"
    )

df_test = (
    df_segments[df_segments["audio_id_base"] == test_audio_id_base]
    .sort_values(["start", "end"])
    .head(10)
    .copy()
)

print("Audio de prueba:", test_audio_file)
print("audio_id_base:", test_audio_id_base)
print("Segmentos de prueba:", len(df_test))

print("\nRuta audio resuelta:")
print(audio_path)

print("\n¿Existe el audio?")
print(audio_path.exists())

audio, sr = load_audio_mono(audio_path)

test_rows = []

for _, row in df_test.iterrows():
    segment_audio = extract_segment_audio(
        audio=audio,
        sr=sr,
        start=float(row["start"]),
        end=float(row["end"])
    )

    text, status = transcribe_audio_array(segment_audio)

    out_row = row.to_dict()
    out_row["text"] = text
    out_row["transcription_status"] = status
    out_row["resolved_audio_path"] = str(audio_path)
    test_rows.append(out_row)

df_test_transcribed = pd.DataFrame(test_rows)

cols_to_show = [
    "audio_file",
    "audio_id_base",
    "start",
    "end",
    "duration",
    "speaker_final",
    "text",
    "transcription_status",
    "resolved_audio_path",
]

cols_to_show = [c for c in cols_to_show if c in df_test_transcribed.columns]

display(df_test_transcribed[cols_to_show])

Audio de prueba: raw_9154117451310006851_clean.wav
audio_id_base: 9154117451310006851
Segmentos de prueba: 10

Ruta audio resuelta:
/home/jupyter/TFM_ProcesadoDeAudios/data/clean_audios/9154117451310006851_clean.wav

¿Existe el audio?
True


,audio_file,audio_id_base,start,end,duration,speaker_final,text,transcription_status,resolved_audio_path
0,raw_9154117451310006851_clean.wav,9154117451310006851,0.030969,4.452219,4.421250,SPEAKER_01,"Hola, buenos días, pregunto por Richard Crelsh...",ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
1,raw_9154117451310006851_clean.wav,9154117451310006851,5.211594,6.342219,1.130625,SPEAKER_00,chiquitame.,ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
2,raw_9154117451310006851_clean.wav,9154117451310006851,6.426594,13.800969,7.374375,SPEAKER_01,"¡Oh, la luz y madalza al lugar del motivo de m...",ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
3,raw_9154117451310006851_clean.wav,9154117451310006851,13.800969,14.965344,1.164375,SPEAKER_00,,empty_transcription,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
4,raw_9154117451310006851_clean.wav,9154117451310006851,15.285969,19.994094,4.708125,SPEAKER_01,tenia más de los técnicos y tendrán más de alg...,ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
5,raw_9154117451310006851_clean.wav,9154117451310006851,20.078469,22.187844,2.109375,SPEAKER_00,"Ya, ahora tenemos internet.",ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
6,raw_9154117451310006851_clean.wav,9154117451310006851,22.727844,23.790969,1.063125,SPEAKER_01,¡Ana tiene que irme!,ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
7,raw_9154117451310006851_clean.wav,9154117451310006851,25.225344,25.967844,0.742500,SPEAKER_01,Bale por...,ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
8,raw_9154117451310006851_clean.wav,9154117451310006851,25.967844,29.899719,3.931875,SPEAKER_00,"La tarda llamada dírenes por la tarvi, más o m...",ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...
9,raw_9154117451310006851_clean.wav,9154117451310006851,31.300344,33.443469,2.143125,SPEAKER_01,"Pues, si es que no nos haría ni chimada.",ok,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...


**Función para transcribir un audio completo**

In [13]:
# =========================
# TRANSCRIPCIÓN DE UN AUDIO COMPLETO
# =========================

def transcribe_one_audio_segments(audio_id_base: str, df_audio_segments: pd.DataFrame):
    """
    Transcribe todos los segmentos de un único audio real normalizado.
    Usa resolve_audio_path para encontrar el audio aunque esté en otra carpeta
    o tenga prefijo raw_.
    """
    start_time = time.time()

    audio_file_ref = str(df_audio_segments["audio_file"].iloc[0])
    audio_path = resolve_audio_path(audio_file_ref, audio_id_base)

    if audio_path is None:
        df_out = df_audio_segments.copy()
        df_out["text"] = ""
        df_out["transcription_status"] = "audio_not_found"
        df_out["resolved_audio_path"] = ""

        summary = {
            "audio_id_base": audio_id_base,
            "audio_file": audio_file_ref,
            "n_segments": len(df_audio_segments),
            "n_ok": 0,
            "n_empty": len(df_audio_segments),
            "n_errors": 0,
            "status": "audio_not_found",
            "elapsed_sec": round(time.time() - start_time, 2)
        }

        return df_out, summary

    audio, sr = load_audio_mono(audio_path)

    rows = []

    for _, row in df_audio_segments.sort_values(["start", "end"]).iterrows():
        segment_audio = extract_segment_audio(
            audio=audio,
            sr=sr,
            start=float(row["start"]),
            end=float(row["end"])
        )

        text, status = transcribe_audio_array(segment_audio)

        out_row = row.to_dict()
        out_row["audio_id_base"] = audio_id_base
        out_row["text"] = text
        out_row["transcription_status"] = status
        out_row["resolved_audio_path"] = str(audio_path)
        rows.append(out_row)

    df_out = pd.DataFrame(rows)

    n_ok = int((df_out["transcription_status"] == "ok").sum())
    n_empty = int(df_out["transcription_status"].isin(["empty_audio", "empty_transcription"]).sum())
    n_errors = int(df_out["transcription_status"].astype(str).str.startswith("error").sum())

    summary = {
        "audio_id_base": audio_id_base,
        "audio_file": audio_file_ref,
        "n_segments": len(df_out),
        "n_ok": n_ok,
        "n_empty": n_empty,
        "n_errors": n_errors,
        "status": "ok",
        "elapsed_sec": round(time.time() - start_time, 2)
    }

    del audio
    gc.collect()

    return df_out, summary


In [14]:
# ============================================================
# REVISIÓN ROBUSTA DE OUTPUTS LOCALES ANTES DEL BATCH
# Usa PER_AUDIO_DIR, que es la carpeta correcta en este notebook
# ============================================================

from pathlib import Path
import pandas as pd

# Asegurar carpeta de outputs por audio
PER_AUDIO_DIR = Path(PER_AUDIO_DIR)
PER_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# Buscar outputs locales ya existentes
output_files_prebatch = sorted(PER_AUDIO_DIR.glob("*_transcribed_segments.csv"))

print("Outputs locales encontrados antes del batch:", len(output_files_prebatch))
print("Carpeta revisada:", PER_AUDIO_DIR)

prebatch_rows = []

for path in output_files_prebatch:
    audio_id_base = get_transcription_csv_audio_id(path)
    
    try:
        df_tmp = pd.read_csv(path)
        
        n_rows = len(df_tmp)
        
        if "transcription_status" in df_tmp.columns:
            n_ok = int((df_tmp["transcription_status"] == "ok").sum())
            n_bad = int((df_tmp["transcription_status"] != "ok").sum())
            statuses = ", ".join(sorted(df_tmp["transcription_status"].dropna().astype(str).unique()))
        else:
            n_ok = n_rows
            n_bad = 0
            statuses = ""
        
        if "text" in df_tmp.columns:
            n_text = int(df_tmp["text"].fillna("").astype(str).str.strip().ne("").sum())
            text_ratio = n_text / n_rows if n_rows > 0 else 0.0
        else:
            n_text = 0
            text_ratio = 0.0
        
        is_bad = (
            n_rows == 0
            or n_ok == 0
            or text_ratio < 0.20
            or "audio_not_found" in statuses
        )
        
        prebatch_rows.append({
            "audio_id_base": audio_id_base,
            "output_path": str(path),
            "n_rows": n_rows,
            "n_ok": n_ok,
            "n_bad": n_bad,
            "n_text": n_text,
            "text_ratio": text_ratio,
            "statuses": statuses,
            "is_bad": is_bad,
            "error": None,
        })
    
    except Exception as e:
        prebatch_rows.append({
            "audio_id_base": audio_id_base,
            "output_path": str(path),
            "n_rows": 0,
            "n_ok": 0,
            "n_bad": 0,
            "n_text": 0,
            "text_ratio": 0.0,
            "statuses": "",
            "is_bad": True,
            "error": str(e),
        })

# Crear DataFrame con columnas garantizadas aunque esté vacío
df_outputs_prebatch = pd.DataFrame(
    prebatch_rows,
    columns=[
        "audio_id_base",
        "output_path",
        "n_rows",
        "n_ok",
        "n_bad",
        "n_text",
        "text_ratio",
        "statuses",
        "is_bad",
        "error",
    ]
)

# Marcar si pertenece al universo objetivo
df_outputs_prebatch["in_target"] = df_outputs_prebatch["audio_id_base"].isin(target_audio_ids)

if len(df_outputs_prebatch) > 0:
    display(
        df_outputs_prebatch
        .groupby(["in_target", "is_bad"])
        .size()
        .reset_index(name="n_files")
    )
    
    display(df_outputs_prebatch.head())
else:
    print("No hay outputs locales antes del batch.")

# Audios objetivo que NO tienen transcripción local válida
valid_existing_outputs = set(
    df_outputs_prebatch.loc[
        (df_outputs_prebatch["in_target"]) & (~df_outputs_prebatch["is_bad"]),
        "audio_id_base"
    ]
)

missing_outputs = sorted(target_audio_ids - valid_existing_outputs)

print("\nAudios objetivo:", len(target_audio_ids))
print("Audios objetivo con transcripción local válida:", len(valid_existing_outputs))
print("Audios objetivo sin transcripción local válida:", len(missing_outputs))

# Esto es lo que debe procesar el batch
target_audio_ids_pending = set(missing_outputs)

print("Audios pendientes para procesar:", len(target_audio_ids_pending))


Outputs locales encontrados antes del batch: 1181
Carpeta revisada: /home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/per_audio


,in_target,is_bad,n_files
0,True,False,1181


,audio_id_base,output_path,n_rows,n_ok,n_bad,n_text,text_ratio,statuses,is_bad,error,in_target
0,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,37,36,1,36,0.972973,"empty_transcription, ok",False,None,True
1,9154117551220006851,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,41,41,0,41,1.000000,ok,False,None,True
2,9154127337680006851,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,27,27,0,27,1.000000,ok,False,None,True
3,9154142438160016851,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,46,45,1,45,0.978261,"empty_transcription, ok",False,None,True
4,9154152155960016851,/home/jupyter/TFM_ProcesadoDeAudios/data/trans...,38,37,1,37,0.973684,"empty_transcription, ok",False,None,True



Audios objetivo: 1181
Audios objetivo con transcripción local válida: 1181
Audios objetivo sin transcripción local válida: 0
Audios pendientes para procesar: 0


**BATCH COMPLETO DE TODOS LOS AUDIOS**

In [15]:
# =========================
# BATCH COMPLETO DE TRANSCRIPCIÓN
# =========================

audio_ids_to_process = sorted(df_segments["audio_id_base"].dropna().astype(str).unique())

print("Audios totales a procesar por audio_id_base:", len(audio_ids_to_process))
print("Carpeta de salida por audio:", PER_AUDIO_DIR)

if len(audio_ids_to_process) != EXPECTED_FINAL_AUDIO_IDS:
    print(f"⚠️ OJO: se esperaban {EXPECTED_FINAL_AUDIO_IDS} audios, pero se van a procesar {len(audio_ids_to_process)}.")


Audios totales a procesar por audio_id_base: 1181
Carpeta de salida por audio: /home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/per_audio
⚠️ OJO: se esperaban 1191 audios, pero se van a procesar 1181.


In [16]:
# =========================
# EJECUCIÓN DEL BATCH COMPLETO
# =========================

summary_rows = []

for i, audio_id_base in enumerate(audio_ids_to_process, start=1):
    clear_output(wait=True)

    output_audio_csv = canonical_transcription_path(audio_id_base)

    print(f"Audio {i}/{len(audio_ids_to_process)}")
    print("audio_id_base:", audio_id_base)
    print("Output:", output_audio_csv.name)

    if output_audio_csv.exists():
        print("Ya existe localmente con nombre canónico, se salta transcripción.")
        if UPLOAD_PER_AUDIO_TO_GCS:
            uploaded_uri = upload_file_to_gcs(output_audio_csv, GCS_PER_AUDIO_PREFIX)
            print("Subida/verificación GCS:", uploaded_uri)
        continue

    df_audio_segments = df_segments[df_segments["audio_id_base"] == audio_id_base].copy()

    df_audio_transcribed, summary = transcribe_one_audio_segments(
        audio_id_base=audio_id_base,
        df_audio_segments=df_audio_segments
    )

    df_audio_transcribed.to_csv(output_audio_csv, index=False)
    summary_rows.append(summary)

    print("Guardado local correctamente.")
    print(summary)

    if UPLOAD_PER_AUDIO_TO_GCS:
        uploaded_uri = upload_file_to_gcs(output_audio_csv, GCS_PER_AUDIO_PREFIX)
        print("Subido a GCS:", uploaded_uri)

    del df_audio_transcribed
    gc.collect()

df_summary_new = pd.DataFrame(summary_rows)

print("Batch terminado.")
print("Audios nuevos procesados en esta ejecución:", len(df_summary_new))

display(df_summary_new.head())


Audio 1181/1181
audio_id_base: 9157454659260016851
Output: 9157454659260016851_transcribed_segments.csv
Ya existe localmente con nombre canónico, se salta transcripción.
Subida/verificación GCS: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/per_audio/9157454659260016851_transcribed_segments.csv
Batch terminado.
Audios nuevos procesados en esta ejecución: 0


""


**Crear resumen completo de audios transcritos**

In [17]:
transcribed_files_all = sorted(PER_AUDIO_DIR.glob("*_transcribed_segments.csv"))
transcribed_files = [
    f for f in transcribed_files_all
    if get_transcription_csv_audio_id(f) in target_audio_ids
]

print("Archivos transcritos locales encontrados:", len(transcribed_files_all))
print("Archivos transcritos dentro del universo objetivo:", len(transcribed_files))

summary_rows = []

for f in transcribed_files:
    df_tmp = pd.read_csv(f)

    n_segments = len(df_tmp)
    n_ok = int((df_tmp["transcription_status"] == "ok").sum()) if "transcription_status" in df_tmp.columns else 0
    n_empty = int(df_tmp["transcription_status"].isin(["empty_audio", "empty_transcription"]).sum()) if "transcription_status" in df_tmp.columns else 0
    n_errors = int(df_tmp["transcription_status"].astype(str).str.startswith("error").sum()) if "transcription_status" in df_tmp.columns else 0

    audio_id_base = get_transcription_csv_audio_id(f)

    summary_rows.append({
        "audio_id_base": audio_id_base,
        "audio_file": df_tmp["audio_file"].iloc[0] if n_segments > 0 and "audio_file" in df_tmp.columns else f.name,
        "transcription_csv": f.name,
        "n_segments": n_segments,
        "n_ok": n_ok,
        "n_empty": n_empty,
        "n_errors": n_errors,
        "ok_ratio": round(n_ok / n_segments, 4) if n_segments > 0 else 0
    })

df_transcription_summary = pd.DataFrame(summary_rows).sort_values("audio_id_base").reset_index(drop=True)

df_transcription_summary.to_csv(TRANSCRIPTION_SUMMARY_CSV, index=False)

print("Resumen guardado en:")
print(TRANSCRIPTION_SUMMARY_CSV)
print("Audios únicos en resumen:", df_transcription_summary["audio_id_base"].nunique())

display(df_transcription_summary.head())
display(df_transcription_summary.describe(include="all"))

if UPLOAD_FINALS_TO_GCS:
    print("Subiendo resumen a GCS:", upload_file_to_gcs(TRANSCRIPTION_SUMMARY_CSV, GCS_OUTPUT_PREFIX))


Archivos transcritos locales encontrados: 1181
Archivos transcritos dentro del universo objetivo: 1181
Resumen guardado en:
/home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/transcription_summary.csv
Audios únicos en resumen: 1181


,audio_id_base,audio_file,transcription_csv,n_segments,n_ok,n_empty,n_errors,ok_ratio
0,9154117451310006851,raw_9154117451310006851_clean.wav,9154117451310006851_transcribed_segments.csv,37,36,1,0,0.9730
1,9154117551220006851,raw_9154117551220006851_clean.wav,9154117551220006851_transcribed_segments.csv,41,41,0,0,1.0000
2,9154127337680006851,raw_9154127337680006851_clean.wav,9154127337680006851_transcribed_segments.csv,27,27,0,0,1.0000
3,9154142438160016851,raw_9154142438160016851_clean.wav,9154142438160016851_transcribed_segments.csv,46,45,1,0,0.9783
4,9154152155960016851,raw_9154152155960016851_clean.wav,9154152155960016851_transcribed_segments.csv,38,37,1,0,0.9737


,audio_id_base,audio_file,transcription_csv,n_segments,n_ok,n_empty,n_errors,ok_ratio
count,1181,1181,1181,1181.000000,1181.000000,1181.000000,1181.0,1181.000000
unique,1181,1181,1181,NaN,NaN,NaN,NaN,NaN
top,9154117451310006851,raw_9154117451310006851_clean.wav,9154117451310006851_transcribed_segments.csv,NaN,NaN,NaN,NaN,NaN
freq,1,1,1,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,34.167655,33.668925,0.498730,0.0,0.984740
std,NaN,NaN,NaN,11.278704,11.209089,0.783975,0.0,0.026063
min,NaN,NaN,NaN,8.000000,8.000000,0.000000,0.0,0.750000
25%,NaN,NaN,NaN,26.000000,25.000000,0.000000,0.0,0.973000
50%,NaN,NaN,NaN,33.000000,33.000000,0.000000,0.0,1.000000
75%,NaN,NaN,NaN,42.000000,41.000000,1.000000,0.0,1.000000


Subiendo resumen a GCS: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/transcription_summary.csv


**Consolidar todos los segmentos transcritos**

In [18]:
transcribed_files_all = sorted(PER_AUDIO_DIR.glob("*_transcribed_segments.csv"))
transcribed_files = [
    f for f in transcribed_files_all
    if get_transcription_csv_audio_id(f) in target_audio_ids
]

print("Archivos por audio encontrados en universo objetivo:", len(transcribed_files))

dfs = []

for f in transcribed_files:
    df_tmp = pd.read_csv(f)
    df_tmp["transcription_csv"] = f.name
    df_tmp["audio_id_base"] = get_transcription_csv_audio_id(f)
    dfs.append(df_tmp)

df_transcribed = pd.concat(dfs, ignore_index=True)

df_transcribed = df_transcribed.sort_values(
    ["audio_id_base", "start", "end"]
).reset_index(drop=True)

# Normalizamos texto
df_transcribed["text"] = df_transcribed["text"].fillna("").astype(str).str.strip()

# Guardamos CSV global
df_transcribed.to_csv(TRANSCRIBED_SEGMENTS_CSV, index=False)

print("CSV global creado en:")
print(TRANSCRIBED_SEGMENTS_CSV)

print("\nDimensiones:", df_transcribed.shape)
print("Audios únicos por audio_file:", df_transcribed["audio_file"].nunique() if "audio_file" in df_transcribed.columns else "N/A")
print("Audios únicos por audio_id_base:", df_transcribed["audio_id_base"].nunique())
print("Segmentos totales:", len(df_transcribed))

if df_transcribed["audio_id_base"].nunique() != EXPECTED_FINAL_AUDIO_IDS:
    print(f"⚠️ OJO: se esperaban {EXPECTED_FINAL_AUDIO_IDS} audios transcritos, pero hay {df_transcribed['audio_id_base'].nunique()}.")

display(df_transcribed.head(20))

if UPLOAD_FINALS_TO_GCS:
    print("Subiendo CSV global a GCS:", upload_file_to_gcs(TRANSCRIBED_SEGMENTS_CSV, GCS_OUTPUT_PREFIX))


Archivos por audio encontrados en universo objetivo: 1181
CSV global creado en:
/home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/all_segments_transcribed.csv

Dimensiones: (40352, 31)
Audios únicos por audio_file: 1181
Audios únicos por audio_id_base: 1181
Segmentos totales: 40352
⚠️ OJO: se esperaban 1191 audios transcritos, pero hay 1181.


,segment_id_raw,audio_file,audio_stem,start,end,duration,speaker,rms_dbfs,overlap_duration_sec,overlap_ratio,...,dist_SPEAKER_01,merged_n_segments,segment_ids_raw,source_csv,audio_id_base,resolved_audio_path,audio_exists_local,text,transcription_status,transcription_csv
0,1,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,0.030969,4.452219,4.421250,SPEAKER_01,-28.919975,0.000000,0.000000,...,0.145629,1,1,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,"Hola, buenos días, pregunto por Richard Crelsh...",ok,9154117451310006851_transcribed_segments.csv
1,2,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,5.211594,6.342219,1.130625,SPEAKER_00,-33.883114,0.000000,0.000000,...,0.814511,1,2,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,chiquitame.,ok,9154117451310006851_transcribed_segments.csv
2,3,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,6.426594,13.800969,7.374375,SPEAKER_01,-29.186810,0.000000,0.000000,...,0.071406,1,3,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,"¡Oh, la luz y madalza al lugar del motivo de m...",ok,9154117451310006851_transcribed_segments.csv
3,4,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,13.800969,14.965344,1.164375,SPEAKER_00,-36.389011,0.118125,0.101449,...,0.975324,1,4,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154117451310006851_transcribed_segments.csv
4,5,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,15.285969,19.994094,4.708125,SPEAKER_01,-27.753717,0.000000,0.000000,...,0.203869,1,5,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,tenia más de los técnicos y tendrán más de alg...,ok,9154117451310006851_transcribed_segments.csv
5,8,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,20.078469,22.187844,2.109375,SPEAKER_00,-24.771809,0.691875,0.328000,...,0.897953,1,8,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,"Ya, ahora tenemos internet.",ok,9154117451310006851_transcribed_segments.csv
6,9,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,22.727844,23.790969,1.063125,SPEAKER_01,-31.448792,0.000000,0.000000,...,0.476506,1,9,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,¡Ana tiene que irme!,ok,9154117451310006851_transcribed_segments.csv
7,11,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,25.225344,25.967844,0.742500,SPEAKER_01,-30.408852,0.050625,0.068182,...,0.572456,1,11,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,Bale por...,ok,9154117451310006851_transcribed_segments.csv
8,12,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,25.967844,29.899719,3.931875,SPEAKER_00,-30.418102,1.080000,0.274678,...,0.678089,1,12,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,La tarea llamada es... Dierne por la tarvi.,ok,9154117451310006851_transcribed_segments.csv
9,13,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,31.300344,33.443469,2.143125,SPEAKER_01,-26.843788,0.000000,0.000000,...,0.306898,1,13,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,"Pues, si es que no nos haría ni chimada.",ok,9154117451310006851_transcribed_segments.csv


Subiendo CSV global a GCS: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/all_segments_transcribed.csv


**Control de calidad básico + ejemplos**

In [19]:
import pandas as pd
from pathlib import Path

print("INICIANDO DIAGNÓSTICO SEGURO DEL NOTEBOOK 06")
print("=" * 80)

# 1. Recoger todos los DataFrames que existen ahora mismo en memoria
dfs = {
    name: obj
    for name, obj in list(globals().items())
    if isinstance(obj, pd.DataFrame)
}

print("\nDATAFRAMES EN MEMORIA:")
for name, df in sorted(dfs.items(), key=lambda x: x[1].shape[0], reverse=True):
    print(f"- {name}: {df.shape[0]} filas, {df.shape[1]} columnas")

# 2. Seleccionar automáticamente el DataFrame de segmentos original
if "df_segments" in dfs:
    df_segments_base = dfs["df_segments"]
    print("\nUsando df_segments como tabla original de segmentos.")
else:
    df_segments_base = None
    print("\nNo encuentro df_segments en memoria.")

# 3. Seleccionar automáticamente el DataFrame transcrito
if "df_transcribed" in dfs:
    df_transcribed_segments = dfs["df_transcribed"]
    selected_transcribed_name = "df_transcribed"
elif "df_transcribed_segments" in dfs:
    df_transcribed_segments = dfs["df_transcribed_segments"]
    selected_transcribed_name = "df_transcribed_segments"
else:
    # Buscar candidato con muchas filas y columnas relacionadas con audio/texto
    candidates = []

    for name, df in dfs.items():
        cols_lower = [str(c).lower() for c in df.columns]

        has_audio = any("audio" in c for c in cols_lower)
        has_text = any(
            ("text" in c)
            or ("transcript" in c)
            or ("transcription" in c)
            for c in cols_lower
        )

        score = 0
        if has_audio:
            score += 10
        if has_text:
            score += 20
        score += df.shape[0] / 100000
        score += df.shape[1] / 1000

        candidates.append((score, name, df))

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    if len(candidates) == 0:
        raise ValueError("No encuentro ningún DataFrame candidato para la transcripción.")

    selected_transcribed_name = candidates[0][1]
    df_transcribed_segments = candidates[0][2]

print(f"\nUsando {selected_transcribed_name} como tabla final transcrita.")
print("Filas tabla transcrita:", len(df_transcribed_segments))
print("Columnas tabla transcrita:", df_transcribed_segments.shape[1])

# 4. Funciones seguras para encontrar columnas
def find_audio_col(df):
    cols = list(df.columns)

    priority = [
        "audio_file",
        "file",
        "filename",
        "audio",
        "audio_path",
        "path"
    ]

    for p in priority:
        for c in cols:
            if str(c).lower() == p:
                return c

    for c in cols:
        if "audio" in str(c).lower():
            return c

    return None


def find_text_col(df):
    cols = list(df.columns)

    priority = [
        "text",
        "transcription",
        "transcription_text",
        "transcribed_text",
        "segment_text"
    ]

    for p in priority:
        for c in cols:
            if str(c).lower() == p:
                return c

    for c in cols:
        c_lower = str(c).lower()
        if "text" in c_lower or "transcript" in c_lower or "transcription" in c_lower:
            return c

    return None


audio_col_transcribed = find_audio_col(df_transcribed_segments)
text_col = find_text_col(df_transcribed_segments)

print("\nCOLUMNAS DETECTADAS:")
print("Columna audio en transcripción:", audio_col_transcribed)
print("Columna texto en transcripción:", text_col)

if audio_col_transcribed is None:
    print("\nNo he podido detectar columna de audio. Columnas disponibles:")
    print(df_transcribed_segments.columns.tolist())

if text_col is None:
    print("\nNo he podido detectar columna de texto. Columnas disponibles:")
    print(df_transcribed_segments.columns.tolist())

# 5. Comparación con df_segments si existe
print("\nCOBERTURA:")
print("Filas en tabla transcrita:", len(df_transcribed_segments))

if df_segments_base is not None:
    print("Filas en df_segments original:", len(df_segments_base))

    audio_col_segments = find_audio_col(df_segments_base)

    print("Columna audio en df_segments:", audio_col_segments)

    if audio_col_segments is not None and audio_col_transcribed is not None:
        expected_audios = set(df_segments_base[audio_col_segments].dropna().unique())
        processed_audios = set(df_transcribed_segments[audio_col_transcribed].dropna().unique())

        missing_audios = sorted(expected_audios - processed_audios)

        print("Audios esperados:", len(expected_audios))
        print("Audios en tabla transcrita:", len(processed_audios))
        print("Audios faltantes:", len(missing_audios))

        if len(missing_audios) > 0:
            print("Primeros audios faltantes:")
            print(missing_audios[:20])
        else:
            print("No faltan audios.")
else:
    print("No puedo comparar contra df_segments porque no está en memoria.")

# 6. Control de textos vacíos
print("\nCALIDAD DE TEXTO:")

if text_col is not None:
    empty_mask = (
        df_transcribed_segments[text_col].isna()
        | (df_transcribed_segments[text_col].astype(str).str.strip() == "")
    )

    n_total = len(df_transcribed_segments)
    n_empty = int(empty_mask.sum())
    n_ok = int((~empty_mask).sum())
    empty_ratio = n_empty / n_total if n_total > 0 else 0

    print("Segmentos totales:", n_total)
    print("Segmentos con texto:", n_ok)
    print("Segmentos vacíos:", n_empty)
    print("Ratio vacíos:", round(empty_ratio * 100, 2), "%")

    if audio_col_transcribed is not None:
        print(
            "Audios con algún segmento vacío:",
            df_transcribed_segments.loc[empty_mask, audio_col_transcribed].nunique()
        )

    print("\nEjemplo de segmentos vacíos:")
    display(df_transcribed_segments.loc[empty_mask].head(10))
else:
    print("No puedo calcular textos vacíos porque no he detectado columna de texto.")

# 7. Revisar resumen por audio si existe
print("\nRESUMEN POR AUDIO:")

if "df_transcription_summary" in dfs:
    df_summary_check = dfs["df_transcription_summary"]
    print("df_transcription_summary encontrado.")
    print("Filas:", len(df_summary_check))
    print("Columnas:", df_summary_check.columns.tolist())

    status_cols = [
        c for c in df_summary_check.columns
        if "status" in str(c).lower() or "error" in str(c).lower()
    ]

    if len(status_cols) > 0:
        print("\nColumnas de estado/error detectadas:", status_cols)

        for col in status_cols:
            print(f"\nValores en {col}:")
            display(df_summary_check[col].value_counts(dropna=False).head(20))

        error_rows = df_summary_check[
            df_summary_check[status_cols]
            .astype(str)
            .apply(lambda row: row.str.contains("error|fail|exception", case=False, na=False).any(), axis=1)
        ]

        print("Filas con posible error:", len(error_rows))
        display(error_rows.head(20))
    else:
        print("No veo columnas claras de error/status en df_transcription_summary.")
else:
    print("No encuentro df_transcription_summary en memoria.")

# 8. Revisar CSVs individuales si existe PER_AUDIO_DIR
print("\nARCHIVOS INDIVIDUALES PER_AUDIO:")

if "PER_AUDIO_DIR" in globals():
    per_audio_path = Path(PER_AUDIO_DIR)

    print("PER_AUDIO_DIR:", per_audio_path)

    if per_audio_path.exists():
        per_audio_csvs = sorted(per_audio_path.glob("*_transcribed_segments.csv"))
        print("CSVs individuales encontrados:", len(per_audio_csvs))
    else:
        print("La carpeta PER_AUDIO_DIR no existe en disco.")
else:
    print("No encuentro la variable PER_AUDIO_DIR.")

print("\nDIAGNÓSTICO TERMINADO")
print("=" * 80)

INICIANDO DIAGNÓSTICO SEGURO DEL NOTEBOOK 06

DATAFRAMES EN MEMORIA:
- df_segments: 40352 filas, 28 columnas
- df_available: 40352 filas, 28 columnas
- df_transcribed: 40352 filas, 31 columnas
- df_local_audit: 1181 filas, 12 columnas
- df_good: 1181 filas, 12 columnas
- keep_rows: 1181 filas, 12 columnas
- df_local_after: 1181 filas, 12 columnas
- df_gcs_audit: 1181 filas, 6 columnas
- df_outputs_prebatch: 1181 filas, 11 columnas
- df_transcription_summary: 1181 filas, 8 columnas
- df_tmp: 27 filas, 31 columnas
- df_test: 10 filas, 28 columnas
- df_test_transcribed: 10 filas, 30 columnas
- missing_audio: 0 filas, 2 columnas
- to_delete: 0 filas, 12 columnas
- df_summary_new: 0 filas, 0 columnas

Usando df_segments como tabla original de segmentos.

Usando df_transcribed como tabla final transcrita.
Filas tabla transcrita: 40352
Columnas tabla transcrita: 31

COLUMNAS DETECTADAS:
Columna audio en transcripción: audio_file
Columna texto en transcripción: text

COBERTURA:
Filas en tabla 

,segment_id_raw,audio_file,audio_stem,start,end,duration,speaker,rms_dbfs,overlap_duration_sec,overlap_ratio,...,dist_SPEAKER_01,merged_n_segments,segment_ids_raw,source_csv,audio_id_base,resolved_audio_path,audio_exists_local,text,transcription_status,transcription_csv
3,4,raw_9154117451310006851_clean.wav,raw_9154117451310006851_clean,13.800969,14.965344,1.164375,SPEAKER_00,-36.389011,0.118125,0.101449,...,0.975324,1,4,raw_9154117451310006851_clean_final_merged.csv,9154117451310006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154117451310006851_transcribed_segments.csv
145,62,raw_9154142438160016851_clean.wav,raw_9154142438160016851_clean,243.030969,243.857844,0.826875,SPEAKER_01,-30.729984,0.000000,0.000000,...,0.538823,1,62,raw_9154142438160016851_clean_final_merged.csv,9154142438160016851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154142438160016851_transcribed_segments.csv
170,43,raw_9154152155960016851_clean.wav,raw_9154152155960016851_clean,85.334094,93.653469,8.319375,SPEAKER_00,-30.390011,0.826875,0.110860,...,0.631500,2,"43,45",raw_9154152155960016851_clean_final_merged.csv,9154152155960016851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154152155960016851_transcribed_segments.csv
255,49,raw_9154202749900006851_clean.wav,raw_9154202749900006851_clean,106.596594,108.182844,1.586250,SPEAKER_01,-30.755577,0.000000,0.000000,...,0.477795,1,49,raw_9154202749900006851_clean_final_merged.csv,9154202749900006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154202749900006851_transcribed_segments.csv
303,97,raw_9154219744450006851_clean.wav,raw_9154219744450006851_clean,272.967219,273.962844,0.995625,SPEAKER_01,-36.342915,0.000000,0.000000,...,0.686490,1,97,raw_9154219744450006851_clean_final_merged.csv,9154219744450006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154219744450006851_transcribed_segments.csv
356,7,raw_9154281474720016851_clean.wav,raw_9154281474720016851_clean,17.260344,17.985969,0.725625,SPEAKER_00,-30.023937,0.320625,0.441860,...,0.932153,1,7,raw_9154281474720016851_clean_final_merged.csv,9154281474720016851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154281474720016851_transcribed_segments.csv
416,19,raw_9154291167650006851_clean.wav,raw_9154291167650006851_clean,58.064094,65.337219,7.273125,SPEAKER_00,-32.014545,0.000000,0.000000,...,0.649327,1,19,raw_9154291167650006851_clean_final_merged.csv,9154291167650006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154291167650006851_transcribed_segments.csv
428,34,raw_9154291167650006851_clean.wav,raw_9154291167650006851_clean,122.222844,123.319719,1.096875,SPEAKER_00,-24.631710,0.000000,0.000000,...,0.455614,1,34,raw_9154291167650006851_clean_final_merged.csv,9154291167650006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154291167650006851_transcribed_segments.csv
433,42,raw_9154291167650006851_clean.wav,raw_9154291167650006851_clean,133.090344,134.744094,1.653750,SPEAKER_00,-25.332260,0.000000,0.000000,...,0.539571,1,42,raw_9154291167650006851_clean_final_merged.csv,9154291167650006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154291167650006851_transcribed_segments.csv
648,11,raw_9154369709820006851_clean.wav,raw_9154369709820006851_clean,50.335344,54.047844,3.712500,SPEAKER_01,-30.735386,0.000000,0.000000,...,0.746820,1,11,raw_9154369709820006851_clean_final_merged.csv,9154369709820006851,/home/jupyter/TFM_ProcesadoDeAudios/data/clean...,True,,empty_transcription,9154369709820006851_transcribed_segments.csv



RESUMEN POR AUDIO:
df_transcription_summary encontrado.
Filas: 1181
Columnas: ['audio_id_base', 'audio_file', 'transcription_csv', 'n_segments', 'n_ok', 'n_empty', 'n_errors', 'ok_ratio']

Columnas de estado/error detectadas: ['n_errors']

Valores en n_errors:


n_errors
0    1181
Name: count, dtype: int64

Filas con posible error: 0


,audio_id_base,audio_file,transcription_csv,n_segments,n_ok,n_empty,n_errors,ok_ratio



ARCHIVOS INDIVIDUALES PER_AUDIO:
PER_AUDIO_DIR: /home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/per_audio
CSVs individuales encontrados: 1181

DIAGNÓSTICO TERMINADO


In [20]:
# =========================
# CONTROL DE CALIDAD DE TRANSCRIPCIÓN
# =========================

df_transcribed["text_clean"] = df_transcribed["text"].fillna("").astype(str).str.strip()
df_transcribed["n_chars"] = df_transcribed["text_clean"].str.len()
df_transcribed["n_words"] = df_transcribed["text_clean"].apply(lambda x: len(x.split()) if x else 0)

quality_summary = {
    "n_segments": len(df_transcribed),
    "n_audios": df_transcribed["audio_file"].nunique(),
    "segments_ok": int((df_transcribed["transcription_status"] == "ok").sum()),
    "segments_empty_text": int((df_transcribed["n_chars"] == 0).sum()),
    "segments_with_text": int((df_transcribed["n_chars"] > 0).sum()),
    "empty_text_ratio": round((df_transcribed["n_chars"] == 0).mean(), 4),
    "avg_chars": round(df_transcribed["n_chars"].mean(), 2),
    "avg_words": round(df_transcribed["n_words"].mean(), 2),
    "median_words": round(df_transcribed["n_words"].median(), 2),
}

quality_summary

{'n_segments': 40352,
 'n_audios': 1181,
 'segments_ok': 39763,
 'segments_empty_text': 589,
 'segments_with_text': 39763,
 'empty_text_ratio': np.float64(0.0146),
 'avg_chars': np.float64(69.71),
 'avg_words': np.float64(13.74),
 'median_words': np.float64(10.0)}

In [21]:
# =========================
# MUESTRA ALEATORIA DE TRANSCRIPCIONES
# =========================

cols_review = [
    "audio_file",
    "start",
    "end",
    "duration",
    "speaker_final",
    "text",
    "transcription_status"
]

display(
    df_transcribed[cols_review]
    .sample(min(30, len(df_transcribed)), random_state=42)
)

,audio_file,start,end,duration,speaker_final,text,transcription_status
33413,raw_bajas_9157287334490016851_clean.wav,31.367844,32.211594,0.843750,SPEAKER_00,Debe.,ok
4201,raw_9155993306570006851_clean.wav,32.835969,37.476594,4.640625,SPEAKER_01,"Yo sé que no hay stopper, yo te veo igual, val...",ok
17590,raw_bajas_9156580017260016851_clean.wav,196.405344,199.544094,3.138750,SPEAKER_01,"en ese alguno tradu, alguno otra consulta la q...",ok
13458,raw_bajas_9156440532000016851_clean.wav,197.299719,198.008469,0.708750,SPEAKER_00,¿Qué no pasa?,ok
38724,raw_bajas_9157393181540016851_clean.wav,64.392219,68.138469,3.746250,SPEAKER_01,el que estoy llamando al teléfono que manda lo...,ok
18906,raw_bajas_9156666325530006851_clean.wav,19.842219,35.991594,16.149375,SPEAKER_01,"y tiene contratado en la segunda residencia, t...",ok
21530,raw_bajas_9156795576590006851_clean.wav,48.512844,52.208469,3.695625,SPEAKER_00,Es una tita que te pide a filmar por cambio de...,ok
39654,raw_bajas_9157425346080036851_clean.wav,191.022219,191.781594,0.759375,SPEAKER_01,para la piel.,ok
6002,raw_bajas_9156070458850006851_clean.wav,181.184094,183.647844,2.463750,SPEAKER_01,"No, no, yo soy instrumental.",ok
9211,raw_bajas_9156260564990026851_clean.wav,212.318469,214.056594,1.738125,SPEAKER_00,"No, no, no, no, no...",ok


In [22]:
FINAL_TRANSCRIPTION_PATH = TRANSCRIPTION_DIR / "06_transcribed_segments_final.csv"
FINAL_SUMMARY_PATH = TRANSCRIPTION_DIR / "06_transcription_summary_final.csv"

df_transcribed.to_csv(FINAL_TRANSCRIPTION_PATH, index=False)
df_transcription_summary.to_csv(FINAL_SUMMARY_PATH, index=False)

print("Guardado final transcripciones en:")
print(FINAL_TRANSCRIPTION_PATH)

print("\nGuardado resumen en:")
print(FINAL_SUMMARY_PATH)

print("\nResumen final corregido:")
print("Audios únicos por audio_id_base:", df_transcribed["audio_id_base"].nunique())
print("Segmentos transcritos:", len(df_transcribed))
print("Archivos por audio en resumen:", df_transcription_summary["audio_id_base"].nunique())

if UPLOAD_FINALS_TO_GCS:
    print("Subiendo finales a GCS...")
    print("- transcripción final:", upload_file_to_gcs(FINAL_TRANSCRIPTION_PATH, GCS_OUTPUT_PREFIX))
    print("- resumen final:", upload_file_to_gcs(FINAL_SUMMARY_PATH, GCS_OUTPUT_PREFIX))


Guardado final transcripciones en:
/home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/06_transcribed_segments_final.csv

Guardado resumen en:
/home/jupyter/TFM_ProcesadoDeAudios/data/transcription_outputs/06_transcription_summary_final.csv

Resumen final corregido:
Audios únicos por audio_id_base: 1181
Segmentos transcritos: 40352
Archivos por audio en resumen: 1181
Subiendo finales a GCS...
- transcripción final: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/06_transcribed_segments_final.csv
- resumen final: gs://catedras_audio_detection/pipelineA/procesados_UNAV/transcription_outputs/06_transcription_summary_final.csv
